# Claims Analysis
Column charts of claims over time (by `loss_date` month) and of other claim variables over time: line, peril, status, incurred / paid / recoveries, average severity and reporting delay.

Note: claims are valued at 31 Dec 2024, so recent months are undeveloped (IBNR/IBNER), and Dec 2020 is a partial month.

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

claims = pd.read_parquet("../data/claims.parquet")
claims["loss_date"] = pd.to_datetime(claims["loss_date"])
claims["report_date"] = pd.to_datetime(claims["report_date"])
claims["loss_month"] = claims["loss_date"].dt.to_period("M").dt.to_timestamp()
claims["net_incurred"] = claims["incurred_to_date"] - claims["recoveries"]
claims["report_delay_days"] = (claims["report_date"] - claims["loss_date"]).dt.days

print(claims.shape)
print(claims["loss_date"].min(), "to", claims["loss_date"].max())

def col_chart(df, title, ylabel, stacked=False, figsize=(16, 5)):
    """Column chart with month labels on the x-axis."""
    ax = df.plot(kind="bar", stacked=stacked, figsize=figsize, width=0.85)
    ax.set_xticklabels([d.strftime("%Y-%m") for d in df.index], rotation=90, fontsize=8)
    ax.set_title(title); ax.set_xlabel("Loss month"); ax.set_ylabel(ylabel)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    plt.tight_layout(); plt.show()

In [0]:
monthly_count = claims.groupby("loss_month").size().rename("claims").to_frame()
col_chart(monthly_count, "Claim count by loss month", "Number of claims")

In [0]:
by_line = claims.pivot_table(index="loss_month", columns="line", values="claim_id", aggfunc="count", fill_value=0)
col_chart(by_line, "Claim count by loss month and line", "Number of claims", stacked=True)

In [0]:
by_peril = claims.pivot_table(index="loss_month", columns="peril", values="claim_id", aggfunc="count", fill_value=0)
by_peril = by_peril[by_peril.sum().sort_values(ascending=False).index]
col_chart(by_peril, "Claim count by loss month and peril", "Number of claims", stacked=True)

In [0]:
by_status = claims.pivot_table(index="loss_month", columns="status", values="claim_id", aggfunc="count", fill_value=0)
col_chart(by_status, "Claim count by loss month and status", "Number of claims", stacked=True)

In [0]:
amounts = claims.groupby("loss_month")[["incurred_to_date", "paid_to_date", "recoveries"]].sum()
col_chart(amounts, "Incurred, paid and recoveries by loss month", "Amount (CU)")

net_by_line = claims[claims["status"] != "repudiated"].pivot_table(
    index="loss_month", columns="line", values="net_incurred", aggfunc="sum", fill_value=0)
col_chart(net_by_line, "Net incurred (excl. repudiated) by loss month and line", "Amount (CU)", stacked=True)

In [0]:
sev = claims[claims["status"] != "repudiated"].pivot_table(
    index="loss_month", columns="line", values="net_incurred", aggfunc="mean")
col_chart(sev, "Average net incurred per claim (excl. repudiated) by loss month and line", "CU per claim")

In [0]:
delay = claims.pivot_table(index="loss_month", columns="line", values="report_delay_days", aggfunc="mean")
col_chart(delay, "Average reporting delay (days) by loss month and line", "Days")

In [0]:
involvement = claims.groupby("loss_month").agg(
    with_repairer=("repairer_id", lambda s: s.notna().mean() * 100),
    with_third_party=("third_party_id", lambda s: s.notna().mean() * 100),
)
col_chart(involvement, "% of claims with repairer / third party by loss month", "% of claims")